In [ ]:
import collections
import math
import os
import shutil
import pandas as pd
import torch
import torchvision
from torch import nn
from d2l import torch as d2l
from torch.utils.data import random_split

In [ ]:
train_augs = torchvision.transforms.Compose([
    torchvision.transforms.Resize(40),
    torchvision.transforms.RandomResizedCrop(32, scale=(0.64, 1.0), ratio=(1.0, 1.0)),
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.4914, 0.4822, 0.4465],
                                     [0.2023, 0.1994, 0.2010])
])

test_augs = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.4914, 0.4822, 0.4465],
                                     [0.2023, 0.1994, 0.2010])
])

train_data = torchvision.datasets.CIFAR10(
    root='./data/cifar-10-python',
    train=True,
    transform=train_augs,
    download=False    # 已经有了，不需要重复下载
)

test_data = torchvision.datasets.CIFAR10(
    root='./data/cifar-10-python',
    train=False,
    transform=test_augs,
    download=False
)

# 50000 张训练图，拿 5000 做验证，45000 做训练
train_size = 45000
val_size = 5000

train_data_split, val_data = random_split(train_data, [train_size, val_size])

In [ ]:
# 模型
def get_net():
    num_classes = 10
    net = d2l.resnet18(num_classes, 3)  # 种类，通道数
    return net

loss = nn.CrossEntropyLoss(reduction="none")

In [ ]:
# 训练函数
# lr_period, lr_decay:每隔多少个epoch，就降低学习率
def train(net, train_iter, valid_iter, num_epochs, lr, wd, devices, lr_period, lr_decay):
    trainer = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.StepLR(trainer, lr_period, lr_decay)
    num_batches, timer = len(train_iter), d2l.Timer()
    legend = ['train loss', 'train acc']
    if valid_iter is not None:
        legend.append('valid acc')
    animator = d2l.Animator(xlabel='epoch', xlim=[1, num_epochs], legend=legend)
    net = nn.DataParallel(net, device_ids=devices).to(devices[0])
    for epoch in range(num_epochs):
        net.train()
        metric = d2l.Accumulator(3)
        for i, (features, labels) in enumerate(train_iter):
            timer.start()
            l, acc = d2l.train_batch_ch13(net, features, labels, loss, trainer, devices)
            metric.add(l, acc, labels.shape[0])
            timer.stop()
            if (i + 1) % (num_batches // 5) == 0 or i == num_batches - 1:
                animator.add(epoch + (i + 1) / num_batches,
                             (metric[0] / metric[2], metric[1] / metric[2]))

        if valid_iter is not None:
            valid_acc = d2l.evaluate_accuracy_gpu(net, valid_iter)
            animator.add(epoch + 1, (None, None, valid_acc))
        scheduler.step()

    measures = f'train loss {metric[0] / metric[2]:.3f}, train acc {metric[1] / metric[2]:.3f}'
    if valid_iter is not None:
        measures += f', valid acc {valid_acc:.3f}'
    print(measures + f'\n{metric[2] * num_epochs} samples processed')

In [ ]:
devices, num_epochs, lr, wd = d2l.try_all_gpus(), 20, 2e-4, 5e-4
lr_period, lr_decay, net = 4, 0.9, get_net()
train(net, train_data_split, val_data, num_epochs, lr, wd, devices, lr_period, lr_decay)

In [ ]:
# 对测试集进行分类并提交结果
net, preds = get_net(), []
train(net, train_data, None, num_epochs, lr, wd, devices, lr_period, lr_decay)

for X, _ in test_data:
    y_hat = net(X.to(devices[0]))
    preds.extend(y_hat.argmax(dim=1).type(torch.int32).cpu().numpy())
    sorted_ids = list(range(1, len(test_data) + 1))
    sorted_ids.sort(key=lambda x:str(x))
    df = pd.DataFrame({'id': sorted_ids, 'label': preds})
    df['label'] = df['label'].apply(lambda x: train_data.classes[x])
    df.to_csv('submission.csv', index=False)